In [14]:
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import requests
import requests
import os
from datetime import datetime
from utils import *
import pyarrow

In [ ]:
ARQUIVO = 'heroes_fabtcg.csv'
url = 'https://cards.fabtcg.com/api/search/v1/cards/?type=Hero&offset=0&limit=99999'

if arquivo_e_atual(ARQUIVO):
    print("O arquivo esta atualizado.")
else:
    print("O arquivo não existe ou esta desatualizado, corrigindo.")
    
    apagar_arquivo(ARQUIVO)
    apagar_arquivo('heroes_fabtcg.parquet')
    
    session = requests.Session()
    session.headers.update(headers)
    
    response = session.get(url)
    response.raise_for_status()

    dados = response.json()
    
    # 1. Ver quais são as chaves principais
    print("Chaves principais:", dados.keys())
    
    lista_cartas = dados.get('data', dados.get('results', dados))
    dict_cartas = {}
    for carta in lista_cartas:
    
        nome_indice = carta["name"]
    
        dict_cartas[nome_indice] = {"tipo" : carta["typebox"].split() }
    
        #remove '-' e 'Hero"
        if "-" in dict_cartas[nome_indice]["tipo"]:
            dict_cartas[nome_indice]["tipo"].remove("-")
        if "Hero" in dict_cartas[nome_indice]["tipo"]:
            dict_cartas[nome_indice]["tipo"].remove("Hero")
    

    dict_final = dict_cartas
    for hero in dict_final:
        dict_final[hero]["Classe"] = []
        dict_final[hero]["Talento"] = []
        dict_final[hero]["Idade"] = []
        dict_final[hero]["Heroi"] = str(hero)
        for item in dict_final[hero]["tipo"]:
            if item in CLASSES_FAB:
                dict_final[hero]["Classe"].append(item)
            if item in TALENTOS_FAB:
                dict_final[hero]["Talento"].append(item)
            if item == "Young":
                dict_final[hero]["Idade"] = [item]
            if "Young" not in dict_final[hero]["Idade"]:
                dict_final[hero]["Idade"] = ["Adult"]
            if len(dict_final[hero]["Talento"]) == 0:
                dict_final[hero]["Talento"] = ["nenhum"]
            if item not in CLASSES_FAB and item not in TALENTOS_FAB and item != "Young":
                print("-----------------------POSSIVEL NOVO TALENTO/CLASSE OU ERRO-----------------------")

        del dict_final[hero]["tipo"]
    
    df = pd.DataFrame(dict_final)
    df = df.T

    
    
    # Salva para a próxima vez
    df.to_parquet('heroes_fabtcg.parquet', index=False)
    df.to_csv('heroes_fabtcg.csv', index=False)
    print("Dados salvos localmente!")

O arquivo não existe ou esta desatualizado, corrigindo.
O arquivo não existe.
Chaves principais: dict_keys(['count', 'next', 'previous', 'results', 'errors'])
Dados salvos localmente!


In [16]:
print(type(df['Classe'].iloc[0])) # <class 'list'>

<class 'list'>


In [19]:
df.head()

,Classe,Talento,Idade,Heroi
Arakni,[Assassin],[nenhum],[Young],Arakni
"Arakni, 5L!p3d 7hRu 7h3 cR4X",[Assassin],[Chaos],[Adult],"Arakni, 5L!p3d 7hRu 7h3 cR4X"
"Arakni, Huntsman",[Assassin],[nenhum],[Adult],"Arakni, Huntsman"
"Arakni, Marionette",[Assassin],[Chaos],[Adult],"Arakni, Marionette"
"Arakni, Solitary Confinement",[Assassin],[nenhum],[Young],"Arakni, Solitary Confinement"
